# CA1 Model Training Notebook (5-Cell Structure)

This notebook mirrors the course CA1 workflow using the project PyTorch pipeline.

| Cell | Course requirement | This project |
|------|-------------------|--------------|
| 1 | Data preparation | Mel-spec PNGs + splits |
| 2 | Model structure | Custom CNN / ResNet50 / MobileNetV2 |
| 3 | compile (loss, optimizer) | CrossEntropyLoss + Adam |
| 4 | fit (epochs, callbacks) | train.py + early stopping |
| 5 | evaluate + plots | evaluate.py + confusion matrix |

In [ ]:
# Cell 1 — Data preparation
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, project_path

cfg = load_config()
train_csv = project_path(cfg["datasets"]["urbansound8k"]["splits_dir"]) / "train_processed.csv"
df = pd.read_csv(train_csv)
print(f"Train clips: {len(df)} | Classes: {df['label'].nunique()}")
print(df['label'].value_counts())

In [ ]:
# Cell 2 — Model structure (3 CA1 models)
import torch
from src.models import build_model, MODEL_NAMES

num_classes = cfg["datasets"]["urbansound8k"]["num_classes"]
for name in MODEL_NAMES:
    model = build_model(name, num_classes=num_classes)
    params = sum(p.numel() for p in model.parameters())
    print(f"{name}: {params:,} parameters")

In [ ]:
# Cell 3 — Loss, optimizer, metrics (model.compile equivalent)
import torch.nn as nn
from torch.optim import Adam

model = build_model("mobilenetv2", num_classes=num_classes)
criterion = nn.CrossEntropyLoss()  # sparse labels (integer class indices)
optimizer = Adam(model.parameters(), lr=cfg["training"]["learning_rate"])
print("Loss:", criterion)
print("Optimizer:", optimizer)
print("Metrics: accuracy + macro recall/F1 via sklearn classification_report")

In [ ]:
# Cell 4 — Training (model.fit equivalent)
# Full training is run via script for reproducibility:
#   python scripts/run_step3_train.py
#   python scripts/run_ca1_ablation_studies.py
print("Run from project root:")
print("  python scripts/run_step3_train.py")
print("  python scripts/run_ca1_ablation_studies.py")

In [ ]:
# Cell 5 — Evaluation results + plots
import json

metrics_path = ROOT / "experiments" / "urbansound8k" / "mobilenetv2" / "test_metrics.json"
with metrics_path.open(encoding="utf-8") as f:
    results = json.load(f)

macro = results["classification_report"]["macro avg"]
print("MobileNetV2 (Model 3) — fold-10 test:")
print(f"  Accuracy:    {results['metrics']['accuracy']:.4f}")
print(f"  Macro recall:{macro['recall']:.4f}")
print(f"  Macro F1:    {results['metrics']['macro_f1']:.4f}")
print("Confusion matrix figure: reports/figures/step3/urbansound8k/mobilenetv2/confusion_matrix_test.png")